# ViTALBench Fig. 3

This notebook redraws Fig. 3 from the canonical dataset files.

The pie chart contains Detection and the three Law categories.


In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUTPUTS = (ROOT / 'paper/Figures/Statistic.pdf', ROOT / 'paper/_aaai/Figures/Statistic.pdf')
detection = json.loads((ROOT / 'detection.json').read_text())
law = json.loads((ROOT / 'law.json').read_text())
law_categories = json.loads((ROOT / 'law_categories.json').read_text())


In [ ]:
detection_counts = Counter(item['type_image'] for item in detection)
law_counts = Counter(item['type_image'] for item in law)
categories = sorted(detection_counts | law_counts)
detection_values = [detection_counts[category] for category in categories]
law_values = [law_counts[category] for category in categories]
# Edit these display names to customize the x-axis and legend.
category_labels = {
    'cong_kenh': 'Cong Kenh',
    'di_vao_duong_cam': 'Di vao duong cam',
    'du_lieu_khong_loi': 'Du lieu khong loi',
    'dung_qua_vach': 'Dung qua vach',
    'khong_mu_bao_hiem': 'Khong mu bao hiem',
    'khong_the_xac_dinh': 'Khong the xac dinh',
    'khong_tuan_thu_den_tin_hieu': 'Khong tuan thu den tin hieu',
    'nguoc_chieu': 'Nguoc chieu',
    'sai_bien_bao': 'Sai bien bao',
    'sai_lan': 'Sai lan',
    'tong_hop': 'Tong hop',
}
labels = [category_labels[category] for category in categories]

category_names = [
    'Violation identification',
    'Regulation Application',
    'Multi-step legal Reasoning',
]
category_values = [sum(item['category'] == name for item in law_categories) for name in category_names]
pie_values = [len(detection), *category_values]
pie_labels = ['Detection', *(f'{name} (Law)' for name in category_names)]
pie_colors = ['#7da8c7', '#80c8bd', '#fa7d73', '#f9b35d']

print({
    'questions': len(detection) + len(law),
    'detection': len(detection),
    'law': len(law),
    'law_categories': dict(zip(category_names, category_values)),
    'images': len({item['image'] for item in detection + law}),
})


In [ ]:
plt.rcParams.update({'font.size': 10, 'axes.titlesize': 14, 'axes.labelsize': 10})
fig, (pie_ax, bar_ax) = plt.subplots(1, 2, figsize=(15, 5.1), gridspec_kw={'width_ratios': [0.85, 1.55]})

total = sum(pie_values)
pie_ax.pie(
    pie_values,
    colors=pie_colors,
    startangle=90,
    counterclock=False,
    autopct=lambda value: f'{value:.1f}%\n({round(value * total / 100):,})',
    textprops={'fontsize': 11, 'fontweight': 'bold'},
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5},
)
pie_ax.set_title('Task / category split', fontweight='bold', pad=14)
pie_ax.legend(pie_labels, loc='lower left', bbox_to_anchor=(-0.02, -0.02), frameon=False, fontsize=9)

x = list(range(len(categories)))
width = 0.28
category_colors = plt.cm.tab20.colors[:len(categories)]
for index, (detection_value, law_value, color) in enumerate(zip(detection_values, law_values, category_colors)):
    bar_ax.bar(index - width / 2, detection_value, width, color=color, edgecolor='#555555', hatch='///', linewidth=0.7)
    bar_ax.bar(index + width / 2, law_value, width, color=color, edgecolor='#555555', linewidth=0.7)
bar_ax.set_title(f'Image category distribution - Detection (n={len(detection):,}) vs Law (n={len(law):,})', fontweight='bold', pad=14)
bar_ax.set_ylabel('count')
bar_ax.set_xticks(x)
bar_ax.set_xticklabels(labels, rotation=32, ha='right', fontsize=9)
bar_ax.grid(axis='y', color='#dddddd', linewidth=0.8)
bar_ax.set_axisbelow(True)
category_handles = [plt.Rectangle((0, 0), 1, 1, facecolor=color, edgecolor='#555555') for color in category_colors]
bar_ax.legend(category_handles, labels, title='Image categories', loc='upper right', frameon=False, fontsize=8, title_fontsize=9)
bar_ax.text(0.99, 0.91, 'Hatched = Detection; solid = Law', transform=bar_ax.transAxes, ha='right', va='top', fontsize=8, color='#555555')
bar_ax.spines['top'].set_visible(False)
bar_ax.spines['right'].set_visible(False)
bar_ax.set_ylim(0, max(law_values) * 1.18)
for index, value in enumerate(detection_values):
    bar_ax.text(index - width / 2, value + 8, f'{value}', ha='center', va='bottom', fontsize=8)
for index, value in enumerate(law_values):
    bar_ax.text(index + width / 2, value + 8, f'{value}', ha='center', va='bottom', fontsize=8)

fig.tight_layout()
for output in OUTPUTS:
    output.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output, format='pdf', bbox_inches='tight')
plt.show()
